## CELL 1: Imports

In [1]:
import os
import cv2
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from tqdm.notebook import tqdm

## CELL 2: Configuration

In [2]:
DATASET_PATH = Path(r'E:\Video_Engagement\OUC-CGE') 
OUTPUT_CSV = 'ouc_cge_5fold_splits.csv'

CLASS_MAPPING = {'high': 'High', 'mid': 'Medium', 'low': 'Low'}

## CELL 3: Scan Dataset and Detect Corrupted Videos

In [3]:
def scan_and_verify_videos(root_path):
    records = []
    print("Scanning and verifying videos. This may take a few minutes...")
    
    for folder_name, class_label in CLASS_MAPPING.items():
        class_dir = root_path / folder_name
        if not class_dir.exists():
            continue
            
        video_files = list(class_dir.rglob('*.mp4'))
        
        for video_path in tqdm(video_files, desc=f"Checking {class_label} class"):
            # Check if video is corrupted
            cap = cv2.VideoCapture(str(video_path))
            ret, frame = cap.read()
            cap.release()
            
            if ret: # If True, the video is readable
                records.append({
                    'video_path': str(video_path),
                    'class_label': class_label
                })
            else:
                print(f"Corrupted file ignored: {video_path.name}")
                
    return pd.DataFrame(records)

df = scan_and_verify_videos(DATASET_PATH)
print(f"\nTotal valid videos found: {len(df)}")
print(df['class_label'].value_counts())

Scanning and verifying videos. This may take a few minutes...


Checking High class:   0%|          | 0/2345 [00:00<?, ?it/s]

Corrupted file ignored: view2521.mp4
Corrupted file ignored: view2630.mp4
Corrupted file ignored: view2631.mp4
Corrupted file ignored: view2632.mp4
Corrupted file ignored: view1316.mp4
Corrupted file ignored: view1319.mp4
Corrupted file ignored: view1322.mp4
Corrupted file ignored: view1325.mp4
Corrupted file ignored: view1328.mp4
Corrupted file ignored: view1331.mp4
Corrupted file ignored: view1334.mp4
Corrupted file ignored: view1337.mp4
Corrupted file ignored: view1340.mp4
Corrupted file ignored: view1343.mp4
Corrupted file ignored: view1346.mp4
Corrupted file ignored: view1349.mp4
Corrupted file ignored: view1352.mp4
Corrupted file ignored: view1355.mp4
Corrupted file ignored: view1358.mp4
Corrupted file ignored: view1361.mp4
Corrupted file ignored: view1364.mp4
Corrupted file ignored: view1367.mp4
Corrupted file ignored: view1370.mp4
Corrupted file ignored: view1373.mp4
Corrupted file ignored: view1376.mp4
Corrupted file ignored: view1379.mp4
Corrupted file ignored: view1382.mp4
C

Checking Medium class:   0%|          | 0/2053 [00:00<?, ?it/s]

Checking Low class:   0%|          | 0/3041 [00:00<?, ?it/s]

Corrupted file ignored: view13.mp4
Corrupted file ignored: view14.mp4
Corrupted file ignored: view33.mp4
Corrupted file ignored: view35.mp4
Corrupted file ignored: view36.mp4
Corrupted file ignored: view37.mp4
Corrupted file ignored: view38.mp4
Corrupted file ignored: view39.mp4
Corrupted file ignored: view40.mp4
Corrupted file ignored: view41.mp4

Total valid videos found: 7364
class_label
Low       3031
High      2280
Medium    2053
Name: count, dtype: int64


## CELL 4 & 5: Split Entire Dataset First, Then Balance Per Fold

In [8]:
from sklearn.model_selection import StratifiedKFold

# Initialize StratifiedKFold (splits the full 7364 videos into 5 chunks)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# List to hold the final assigned rows
all_folds_data = []

X = df['video_path']
y = df['class_label']

print("Generating dynamically balanced folds...")

for fold_idx, (train_index, val_index) in enumerate(skf.split(X, y)):
    # 1. Get the imbalanced train and val sets for this specific fold
    df_train = df.iloc[train_index].copy()
    df_val = df.iloc[val_index].copy()
    
    # 2. Find the minority class count in THIS fold's training set
    # (Since Medium is 2053 total, 4/5 of it will be ~1642 videos for training)
    min_train_count = df_train['class_label'].value_counts().min()
    
    # 3. Randomly sample the majority classes to match the minority class
    # We use 'random_state=fold_idx' so the random selection changes in every fold!
    df_train_balanced = df_train.groupby('class_label').sample(n=min_train_count, random_state=fold_idx)
    df_train_balanced['fold'] = fold_idx
    df_train_balanced['phase'] = 'train'
    
    # 4. Do the same balancing for this fold's validation set 
    # (1/5 of Medium will be ~411 videos for validation)
    min_val_count = df_val['class_label'].value_counts().min()
    df_val_balanced = df_val.groupby('class_label').sample(n=min_val_count, random_state=fold_idx)
    df_val_balanced['fold'] = fold_idx
    df_val_balanced['phase'] = 'val'
    
    # Append to our master list
    all_folds_data.append(df_train_balanced)
    all_folds_data.append(df_val_balanced)

# Combine everything into one final DataFrame
df_dynamic_folds = pd.concat(all_folds_data).reset_index(drop=True)

Generating dynamically balanced folds...


## CELL 6: Save and Verify Results

In [9]:
df_dynamic_folds.to_csv(OUTPUT_CSV, index=False)
print(f"\nSuccessfully saved dynamic 5-Fold splits to {OUTPUT_CSV}")

# Verify what happened in Fold 0 to ensure it worked
fold_0_train = df_dynamic_folds[(df_dynamic_folds['fold'] == 0) & (df_dynamic_folds['phase'] == 'train')]
fold_0_val = df_dynamic_folds[(df_dynamic_folds['fold'] == 0) & (df_dynamic_folds['phase'] == 'val')]

print("\n--- Verification for Fold 0 ---")
print("Training Set Distribution (Perfectly Balanced):")
print(fold_0_train['class_label'].value_counts())
print("\nValidation Set Distribution (Perfectly Balanced):")
print(fold_0_val['class_label'].value_counts())


Successfully saved dynamic 5-Fold splits to ouc_cge_5fold_splits.csv

--- Verification for Fold 0 ---
Training Set Distribution (Perfectly Balanced):
class_label
High      1642
Low       1642
Medium    1642
Name: count, dtype: int64

Validation Set Distribution (Perfectly Balanced):
class_label
High      411
Low       411
Medium    411
Name: count, dtype: int64
